In [4]:
from dotenv import load_dotenv
from openai import OpenAI
import json

load_dotenv()
openai_client = OpenAI()

In [1]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)


In [6]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [15]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
""".strip()


In [5]:

search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [7]:
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == 'search':
        result = search(**args)
    
    result_json = json.dumps(result, indent=2)

    return {
        'type': 'function_call_output',
        'call_id': call.call_id,
        'output': result_json,
    }
        

In [11]:
def agent_loop(instructions, question, model='gpt-5.4-mini') -> str:
    messages = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question},
    ]


    it = 1 

    while True:
        print(f'iteration #{it}...')
        has_function_calls = False 

        response = openai_client.responses.create(
            model = model,
            input = messages,
            tools = [search_tool],
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == 'function_call':
                print('function_call:', item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True 
            elif item.type =='message':
                print('ASSISTANT:')
                print(item.content[0].text)
        
        it += 1 
        if has_function_calls == False:
            break

In [16]:
agent_loop(instructions, "what's queen gambit?")

iteration #1...
function_call: search {"query":"queen gambit"}
iteration #2...
function_call: search {"query":"queen's gambit chess opening"}
iteration #3...
function_call: search {"query":"queen gambit off topic chess"}
iteration #4...
ASSISTANT:
I couldn’t find any course FAQ entry about “queen’s gambit,” so it looks like this is off-topic for the course.

If you have a course-related question, I can help with that. Is there another area you want to explore?
